
# 練習問題: 1つのループで平均と分散を求める (2変数の reduction)

## 目標

`reduction` 節に**複数の変数**を並べることで, 1つのループの中で2つの総和を同時に集計できることを体験する. ここでは配列の和 `s` と二乗和 `sq` を同時に求め, 平均と分散を計算する.

## 課題

`mean_var.cpp` (または `mean_var.f90`) は, 要素数 `n = 1000000` の配列 `a` (`a[i] = sin(i)`) について, 1つのループで

- 和    `s  += a[i]`
- 二乗和 `sq += a[i]*a[i]`

を求め, ループ後に `mean = s/n`, `variance = sq/n - mean*mean` を計算する.

コメント `TODO` の指示に従ってループを並列化せよ. **2つの総和を1つの `reduction` 節にまとめて**指定する.

- C++: ループの直前に `#pragma omp parallel for reduction(+:s,sq)` を加える.
- Fortran: `do` ループを `!$omp parallel do private(x) reduction(+:s,sq)` と `!$omp end parallel do` で囲む (一時変数 `x` はスレッドごとに別々にするため `private` にする).

`reduction(+:s,sq)` のように, カンマで区切って複数の変数を1つの節に並べられる.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mean_var.cpp -o mean_var.exe

# Fortran
nvfortran -fast -mp=multicore mean_var.f90 -o mean_var.exe
```

```
OMP_NUM_THREADS=4 ./mean_var.exe
```

## 期待される結果

平均はほぼ 0, 分散はほぼ 0.5 になる (`sin` の値の性質による). 例:

```
mean = 0.000000, variance = 0.499999
```

`OMP_NUM_THREADS` を変えても結果はほぼ同じになる. ただし浮動小数点の和は**足す順序**によってごくわずかに変わるため, スレッド数を変えると最下位の桁が変動しうることに注意せよ.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mean_var.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:mean_var.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ mean_var.cpp
#include <cstdio>
#include <cmath>

int main() {
  const long n = 1000000L;
  static double a[n];
  for (long i = 0; i < n; i++) {
    a[i] = sin((double)i);
  }
  double s = 0.0, sq = 0.0;
  // TODO: 下のループを #pragma omp parallel for reduction(+:s,sq) で並列化し, 2つの総和の競合を解消せよ.
  for (long i = 0; i < n; i++) {
    double x = a[i];
    s  += x;
    sq += x * x;
  }
  double mean = s / n;
  double var  = sq / n - mean * mean;
  printf("mean = %f, variance = %f\n", mean, var);
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast mean_var.cpp -o mean_var_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./mean_var_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./mean_var_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mean_var_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mean_var.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ mean_var.f90
program mean_var
  integer(8), parameter :: n = 1000000_8
  real(8), allocatable :: a(:)
  real(8) :: s, sq, x, mean, var
  integer(8) :: i
  allocate(a(0:n-1))
  do i = 0, n - 1
     a(i) = sin(real(i, 8))
  end do
  s = 0.0d0
  sq = 0.0d0
  ! TODO: 下のループを !$omp parallel do private(x) reduction(+:s,sq) で並列化し, 2つの総和の競合を解消せよ.
  do i = 0, n - 1
     x = a(i)
     s  = s + x
     sq = sq + x * x
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  mean = s / n
  var  = sq / n - mean * mean
  print "(a,f0.6,a,f0.6)", "mean = ", mean, ", variance = ", var
end program mean_var

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast mean_var.f90 -o mean_var_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./mean_var_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./mean_var_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mean_var_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mean_var.f90}